# Menyiapkan Lingkungan dan Data Contoh

In [1]:
import numpy as np, pandas as pd, os, shutil

def bangkitkan_trips_sintetis(keluar, n=1_000_000, seed=42):
  rng = np.random.default_rng(seed)
  wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
  zona_ids = np.arange(1, 264)
  wilayah_zona = rng.choice(wilayah, size=len(zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05,0.03])
  zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})

  pop = np.clip(rng.zipf(1.6, size=len(zona_ids)).astype(float), 1, 60)
  pop /= pop.sum()
  pu = rng.choice(zona_ids, size=n, p=pop)
  do = rng.choice(zona_ids, size=n, p=pop)

  awal = pd.Timestamp("2023-01-01")
  pickup = awal + pd.to_timedelta(rng.integers(0, 31 * 24 * 60, n), unit="m")
  durasi = rng.integers(2, 90, n)
  dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
  fare = np.round(2.5 + dist * 2.8 + rng.gamma(2.0, 1.0, n), 2)
  tip = np.round(rng.gamma(1.5, 1.2, n), 2)
  pay = rng.choice([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])

  d = pd.DataFrame({
      "tpep_pickup_datetime": pickup,
      "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
      "passenger_count": rng.integers(1, 5, n).astype("float64"),
      "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
      "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
      "total_amount": np.round(fare + tip + 3.0, 2),
      "durasi_menit": durasi.astype("float64"),
      "nama_pembayaran": pd.Series(pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4:"Sengketa"}).values,
      })
  d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID",how="left")
  d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
  d["hujan_mm"] = np.round(rng.gamma(0.4, 0.8, n), 2)
  if os.path.exists(keluar):
    shutil.rmtree(keluar)
  d.to_parquet(keluar, partition_cols=["borough_naik"], index=False)
  return keluar

TRIPS_BERSIH = bangkitkan_trips_sintetis("/content/trips_bersih_sintetis")

In [3]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")
DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
TRIPS_BERSIH = "/content/trips_bersih_sintetis"

import os, time, json, sqlite3, random, shutil, glob
import numpy as np
import pandas as pd


os.makedirs(DIR_SIMPAN, exist_ok=True)

assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Jalankan ulang P3/K-10 ataupakai K-1b."

# Pengukur waktu (dari Praktikum 1/K-3, versi ringkas)
catatan = []
def ukur(label, fungsi):
  t0 = time.perf_counter()
  hasil = fungsi()
  detik = time.perf_counter() - t0
  catatan.append({"langkah": label, "detik": round(detik, 4)})
  print(f"[{label}] {detik:.4f} s")
  return hasil

file_parquet = glob.glob(
    f"{TRIPS_BERSIH}/**/*.parquet",
    recursive=True
)

def baca_semua_parquet():
  hasil = []

  for f in file_parquet:
    temp = pd.read_parquet(f)

    folder = os.path.basename(os.path.dirname(f))
    wilayah = folder.split("=", 1)[1]
    temp["borough_naik"] = wilayah
    hasil.append(temp)

  return pd.concat(hasil, ignore_index=True)

df = ukur("baca trips_bersih", baca_semua_parquet)

# Ganti nama kolom asli data TLC menjadi nama yang lebih mudah dibaca
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}

df = df.rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])
print(df["wilayah_jemput"].value_counts())

# Contoh 300.000 baris untuk K-2 dan K-3 (agar tidak terlalu lama)
sample = df.sample(n=min(300_000, len(df)), random_state=42).reset_index(drop=True)
sample["id_perjalanan"] = sample.index + 1
sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

display(sample.head())

Mounted at /content/drive
[baca trips_bersih] 0.2153 s
baris: 1,000,000 | kolom: 15
wilayah_jemput
Manhattan          391107
Queens             270929
Brooklyn           205487
Staten%20Island     67041
Bronx               60912
EWR                  4524
Name: count, dtype: int64


,waktu_jemput,tpep_dropoff_datetime,passenger_count,jarak_mil,zona_jemput,zona_tujuan,payment_type,fare_amount,tip,total_bayar,durasi_menit,metode_bayar,suhu_c,hujan_mm,wilayah_jemput,id_perjalanan
0,2023-01-21 20:45:00,2023-01-21 21:20:00,4.0,5.17,231,231,1,18.20,0.73,21.93,35.0,Kartu kredit,9.1,0.19,Manhattan,1
1,2023-01-02 16:19:00,2023-01-02 17:06:00,2.0,2.41,160,177,2,14.16,0.24,17.40,47.0,Tunai,5.3,0.00,Queens,2
2,2023-01-29 13:19:00,2023-01-29 14:24:00,4.0,1.14,144,252,1,7.00,2.95,12.95,65.0,Kartu kredit,1.4,0.17,Brooklyn,3
3,2023-01-19 00:39:00,2023-01-19 01:05:00,3.0,2.23,138,183,1,9.20,1.11,13.31,26.0,Kartu kredit,9.3,0.08,Brooklyn,4
4,2023-01-28 10:40:00,2023-01-28 10:47:00,1.0,1.36,143,166,2,8.59,0.66,12.25,7.0,Tunai,5.9,0.17,Staten%20Island,5


# Tabel Biasa vs Key-Value

In [4]:
KOLOM = ["id_perjalanan", "waktu_jemput", "zona_jemput", "zona_tujuan",
         "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data = sample[KOLOM].copy()

# Model 1 - relasional: tabel SQLite
if os.path.exists("uji.db"):
  os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

# Model 2 - key-value: kamus (dict) dengan kunci "perjalanan:<id>", nilainya teks JSON
def bangun_kv():
  return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r) for r in data.to_dict("records")}

kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

# Pertanyaan A: ambil satu perjalanan berdasarkan id (200 id acak)
random.seed(42)
id_uji = random.sample(range(1, len(data) + 1), 200)

def cari_sql():
  cur = con.cursor()
  return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?", (i,)).fetchone()
    for i in id_uji]

def cari_kv():
  return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1 = ukur("cari 200 perjalanan - SQL tanpa indeks", cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)") # buat indeks
r2 = ukur("cari 200 perjalanan - SQL dengan indeks", cari_sql)
r3 = ukur("cari 200 perjalanan - key-value", cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

# Pertanyaan B: hitung rata-rata total bayar per wilayah (melibatkan SEMUA data)
def hitung_sql():
  return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC,
                          1""").fetchall()

def hitung_kv():
  total, jumlah = {}, {}
  for teks in kv.values():
    d = json.loads(teks)      # setiap nilai harus dibuka dulu
    w = d["wilayah_jemput"]
    total[w] = total.get(w, 0) + d["total_bayar"]
    jumlah[w] = jumlah.get(w, 0) + 1
  return sorted([(w, jumlah[w], round(total[w] / jumlah[w], 2)) for w in jumlah],
                  key=lambda x: (-x[1], x[0]))

h1 = ukur("hitung per wilayah - SQL GROUP BY", hitung_sql)
h2 = ukur("hitung per wilayah - key-value (buka semua nilai)", hitung_kv)
print(h1[:3])
print("hasil sama?", h1 == h2)


[relasional: muat tabel] 0.7460 s
[key-value: bangun dict] 1.7486 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 2.6059 s
[cari 200 perjalanan - SQL dengan indeks] 0.0023 s
[cari 200 perjalanan - key-value] 0.0008 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.1239 s
[hitung per wilayah - key-value (buka semua nilai)] 0.8537 s
[('Manhattan', 117529, 18.25), ('Queens', 81445, 18.26), ('Brooklyn', 61379, 18.26)]
hasil sama? True


# Bentuk Data yang Fleksibel

In [5]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
  dok = {
      "id_perjalanan": int(r.id_perjalanan),
      "waktu_jemput": r.waktu_jemput,
      "jemput": {"zona": int(r.zona_jemput), "wilayah": r.wilayah_jemput},
      "tujuan": {"zona": int(r.zona_tujuan)},
      "biaya": {"jarak_mil": float(r.jarak_mil),
                "total": float(r.total_bayar),
                "metode_bayar": r.metode_bayar},
  }
  # Tip hanya dicatat untuk pembayaran kartu -> field ini ada di sebagian dokumen saja
  if r.metode_bayar == "Kartu kredit":
      dok["biaya"]["tip"] = float(r.tip)
  return dok

db = TinyDB(storage=MemoryStorage)
tabel = db.table("perjalanan")
docs = [ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)
print("jumlah dokumen:", len(tabel))
print(json.dumps(tabel.all()[0], indent=2))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-21 20:45:00",
  "jemput": {
    "zona": 231,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 231
  },
  "biaya": {
    "jarak_mil": 5.17,
    "total": 21.93,
    "metode_bayar": "Kartu kredit",
    "tip": 0.73
  }
}


In [6]:
Q = Query()
print("jemput di Manhattan & total > 50 :", len(tabel.search((Q.jemput.wilayah == "Manhattan") &
 (Q.biaya.total > 50))))
print("dokumen yang punya field tip     :", len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip     :", len(tabel.search(~Q.biaya.tip.exists())))

# Menambah field baru ("cuaca") pada satu dokumen - tanpa mengubah struktur apa pun
tabel.insert({"id_perjalanan": 999_999, "waktu_jemput": "2023-01-15 08:00:00",
              "jemput": {"zona": 161, "wilayah": "Manhattan"}, "tujuan": {"zona": 236},
              "biaya": {"jarak_mil": 1.2, "total": 14.5, "metode_bayar": "Kartu kredit", "tip":3.0},
              "cuaca": {"hujan": True, "suhu_c": -2.0}})
print("dokumen yang punya field cuaca   :", len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang           :", len(tabel))

jemput di Manhattan & total > 50 : 7
dokumen yang punya field tip     : 13997
dokumen yang TIDAK punya tip     : 6003
dokumen yang punya field cuaca   : 1
total dokumen sekarang           : 20001


# Penyimpanan per Kolom: DuckDB pada Parquet

In [7]:
import duckdb

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
con_d = duckdb.connect()

# VIEW = "jendela" bernama yang membaca Parquet langsung (tidak menyalin data).
# Di sinilah nama kolom asli TLC diberi nama yang mudah dibaca.
con_d.sql(f"""
    CREATE VIEW perjalanan AS
    SELECT tpep_pickup_datetime AS waktu_jemput,
           PULocationID         AS zona_jemput,
           DOLocationID         AS zona_tujuan,
           borough_naik         AS wilayah_jemput,
           trip_distance        AS jarak_mil,
           total_amount         AS total_bayar,
           nama_pembayaran      AS metode_bayar
    FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL = """SELECT wilayah_jemput, COUNT(*) AS jumlah, ROUND(AVG(total_bayar), 2) AS rata_bayar
         FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC, wilayah_jemput"""
hasil_d = ukur("DuckDB - hitung per wilayah (langsung dari Parquet)", lambda:
con_d.sql(SQL).df())
print(hasil_d)

# Pembanding: SQLite (menyimpan per baris) pada data yang sama
KOLOM_S = ["wilayah_jemput", "total_bayar", "jarak_mil", "zona_jemput", "zona_tujuan"]
if os.path.exists("baris.db"):
  os.remove("baris.db")
con_s = sqlite3.connect("baris.db")
ukur("SQLite - muat seluruh baris ke tabel", lambda: df[KOLOM_S].to_sql("perjalanan", con_s,
index=False))
hasil_s = ukur("SQLite - hitung per wilayah", lambda: con_s.execute(
    """SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar), 2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC, 1""").fetchall())
print("hasil sama?", [tuple(r) for r in hasil_d.itertuples(index=False)] == [tuple(r) for r in
hasil_s])

[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.0231 s
  wilayah_jemput  jumlah  rata_bayar
0      Manhattan  391107       18.26
1         Queens  270929       18.27
2       Brooklyn  205487       18.29
3  Staten Island   67041       18.28
4          Bronx   60912       18.21
5            EWR    4524       18.31
[SQLite - muat seluruh baris ke tabel] 1.3238 s
[SQLite - hitung per wilayah] 0.3600 s
hasil sama? False


In [8]:
# Filter pada kolom partisi: DuckDB hanya membuka folder wilayah yang diminta (partition pruning,P3)
SQL2 = """SELECT COUNT(*), ROUND(AVG(total_bayar), 2)
          FROM perjalanan WHERE wilayah_jemput = 'Staten Island'"""
print(ukur("DuckDB - hanya wilayah Staten Island", lambda: con_d.sql(SQL2).fetchall()))
ukuran_pq = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(TRIPS_BERSIH) for f
in fs) / 1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom):{os.path.getsize('baris.db') / 1024**2:.1f} MB")

[DuckDB - hanya wilayah Staten Island] 0.0088 s
[(67041, 18.28)]
Parquet (15 kolom): 32.2 MB | SQLite (5 kolom):37.8 MB


# Graph: Pertanyaan tentang Keterhubungan

In [9]:
import networkx as nx

# Satu baris = satu arus: berapa perjalanan dari zona jemput ke zona tujuan tertentu
arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan GROUP BY 1, 2""").df())

# Zona = node (titik), arus = edge (garis berarah, bobotnya jumlah perjalanan)
G = nx.DiGraph()
for r in arus.itertuples(index=False):
  G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
print("node (zona):", G.number_of_nodes(), "| edge (arus):", G.number_of_edges())

# Lima arus terbesar. Urutan dibuat pasti dengan pengurut tambahan (jumlah, asal, tujuan)
top_graf = sorted(G.edges(data=True), key=lambda e: (-e[2]["jumlah"], e[0], e[1]))[:5]
print("graph:", [(a, b, d["jumlah"]) for a, b, d in top_graf])
top_sql = arus.sort_values(["jumlah", "asal", "tujuan"], ascending=[False, True, True]).head(5)
print("SQL :", list(top_sql.itertuples(index=False, name=None)))

# Lima zona yang paling banyak menjadi tujuan
populer = sorted(G.in_degree(weight="jumlah"), key=lambda kv: (-kv[1], kv[0]))[:5]
print("zona tujuan terpopuler:", populer)

[hitung arus antar zona] 0.0443 s
node (zona): 263 | edge (arus): 44120
graph: [(46, 205, 789), (146, 129, 787), (74, 221, 765), (221, 166, 765), (46, 221, 761)]
SQL : [(46, 205, 789), (146, 129, 787), (74, 221, 765), (221, 166, 765), (46, 221, 761)]
zona tujuan terpopuler: [(183, 26941), (104, 26916), (146, 26891), (164, 26867), (194, 26847)]


In [10]:
MIN = 50 # abaikan arus yang terlalu jarang (kurang dari 50 perjalanan)
zona_awal = populer[0][0] # mulai dari zona tujuan terpopuler

# Pertanyaan: zona mana yang bisa dicapai dalam TEPAT 2 langkah dari zona_awal?
# (1) cara graph: telusuri tetangga dari tetangga
def dua_langkah_graph():
  hasil = set()
  for b, d1 in G[zona_awal].items():
      if d1["jumlah"] < MIN:
        continue
      for c, d2 in G[b].items():
        if d2["jumlah"] >= MIN and c != zona_awal:hasil.add(c)
  return hasil

# (2) cara SQL: harus menggabungkan tabel arus dengan dirinya sendiri (self-join)
con_d.register("arus", arus)
def dua_langkah_sql():
  return set(con_d.sql(f"""
      SELECT DISTINCT a2.tujuan
      FROM arus a1 JOIN arus a2 ON a1.tujuan = a2.asal
      WHERE a1.asal = {zona_awal} AND a1.jumlah >= {MIN}
        AND a2.jumlah >= {MIN} AND a2.tujuan <> {zona_awal}""").df()["tujuan"])

hg = ukur("2 langkah - graph (NetworkX)", dua_langkah_graph)
hs = ukur("2 langkah - SQL self-join (DuckDB)", dua_langkah_sql)
print("zona awal:", zona_awal, "| terjangkau (graph):", len(hg), "| (SQL):", len(hs), "| sama?",
hg == hs)

# (3) Jumlah langkah bebas: "dalam paling banyak k langkah" - cukup ganti angka k
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])
for k in (1, 2, 3, 4):
  n = len(nx.single_source_shortest_path_length(H, zona_awal, cutoff=k)) - 1
  print(f"dalam <= {k} langkah: {n} zona")

[2 langkah - graph (NetworkX)] 0.0083 s
[2 langkah - SQL self-join (DuckDB)] 0.0101 s
zona awal: 183 | terjangkau (graph): 91 | (SQL): 91 | sama? True
dalam <= 1 langkah: 83 zona
dalam <= 2 langkah: 91 zona
dalam <= 3 langkah: 91 zona
dalam <= 4 langkah: 91 zona


# Indeks Spasial

In [11]:
from shapely import STRtree, box, points

# CATATAN: data TLC 2023 hanya berisi ID zona, TANPA koordinat. Titik di bawah ini SINTETIS.

rng = np.random.default_rng(42)
N = 200_000
bujur = rng.normal(-73.97, 0.06, N).clip(-74.25, -73.70) # longitude
lintang = rng.normal(40.74, 0.05, N).clip(40.50, 40.92)  # latitude
titik = points(bujur, lintang)

# 1.000 wilayah pencarian berupa kotak kecil (sekitar 1 km x 1 km)
M = 1000
cx = rng.uniform(-74.05, -73.85, M)
cy = rng.uniform(40.65, 40.85, M)
kotak = [box(x - 0.005, y - 0.005, x + 0.005, y + 0.005) for x, y in zip(cx, cy)]

# Cara 1: periksa titik satu per satu dengan loop Python (hanya 5 kotak, 1.000 akan sangat lama)
def cara_loop():
  return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]

# Cara 2: bandingkan seluruh titik sekaligus dengan NumPy (tanpa indeks spasial), 1.000 kotak
def cara_numpy():
  hasil = []
  for x, y in zip(cx, cy):
    m = (bujur >= x - 0.005) & (bujur <= x + 0.005) & (lintang >= y - 0.005) & (lintang <= y+ 0.005)
    hasil.append(int(m.sum()))
  return hasil

h_loop = ukur("cara 1 - loop Python, 5 kotak", cara_loop)
h_np = ukur("cara 2 - NumPy, 1000 kotak", cara_numpy)

# Cara 3: indeks spasial STRtree (sejenis R-tree)
pohon = ukur("bangun indeks STRtree", lambda: STRtree(titik))
h_idx = ukur("cara 3 - STRtree, 1000 kotak",
             lambda: [len(pohon.query(k, predicate="contains")) for k in kotak])

print("5 kotak pertama - loop:", h_loop, "| STRtree:", h_idx[:5])
print("hasil NumPy dan STRtree sama?", h_np == h_idx, "| loop dan STRtree sama (5 pertama)?",h_loop == h_idx[:5])
print("total titik ditemukan (1000 kotak):", sum(h_idx))

[cara 1 - loop Python, 5 kotak] 7.8672 s
[cara 2 - NumPy, 1000 kotak] 0.2388 s
[bangun indeks STRtree] 0.0466 s
[cara 3 - STRtree, 1000 kotak] 0.1596 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True | loop dan STRtree sama (5 pertama)? True
total titik ditemukan (1000 kotak): 417545


# Delta Lake: Versi, Time Travel, dan Schema Enforcement

In [12]:
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
data_delta = df[KOLOM_DELTA].sample(n=min(200_000, len(df)),
random_state=42).reset_index(drop=True)

PATH_DELTA = "/content/trips_delta"
if os.path.exists(PATH_DELTA):
  shutil.rmtree(PATH_DELTA)

# Versi 0: menulis data awal (100.000 baris)
write_deltalake(PATH_DELTA, data_delta.iloc[:100_000], mode="overwrite")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 1: menambah 50.000 baris
write_deltalake(PATH_DELTA, data_delta.iloc[100_000:150_000], mode="append")
dt = DeltaTable(PATH_DELTA)
print("versi:", dt.version(), "| baris:", len(dt.to_pandas()))

# Versi 2: koreksi data. Anggap tarif Tunai di wilayah Bronx perlu ditambah 1.0
sekarang = data_delta.iloc[:150_000]
n_koreksi = int(((sekarang["wilayah_jemput"] == "Bronx") & (sekarang["metode_bayar"] =="Tunai")).sum())
print("baris yang akan dikoreksi:", n_koreksi)
dt.update(predicate="wilayah_jemput = 'Bronx' AND metode_bayar = 'Tunai'",
          updates={"total_bayar": "total_bayar + 1.0"})
print("versi:", DeltaTable(PATH_DELTA).version())

versi: 0 | baris: 100000
versi: 1 | baris: 150000
baris yang akan dikoreksi: 2455
versi: 2


In [13]:
# TIME TRAVEL: baca tabel sebagaimana keadaannya pada versi tertentu
v0 = DeltaTable(PATH_DELTA, version=0).to_pandas()
v1 = DeltaTable(PATH_DELTA, version=1).to_pandas()
terbaru = DeltaTable(PATH_DELTA).to_pandas()
print("baris v0:", len(v0), "| v1:", len(v1), "| terbaru:", len(terbaru))
selisih = terbaru["total_bayar"].sum() - v1["total_bayar"].sum()
print("selisih total_bayar (terbaru - v1):", round(selisih, 2), "| jumlah baris dikoreksi:",n_koreksi)

# Riwayat perubahan tabel
for h in DeltaTable(PATH_DELTA).history():
  print(h.get("version"), h.get("operation"))

# Log transaksi: satu file JSON per versi
print(sorted(os.listdir(f"{PATH_DELTA}/_delta_log")))

# PENEGAKAN SKEMA: menambah data dengan kolom asing harus DITOLAK, tabel tidak berubah
tambahan = data_delta.iloc[150_000:150_010].copy()
tambahan["kolom_asing"] = 1
try:
  write_deltalake(PATH_DELTA, tambahan, mode="append")
  print("append kolom asing: BERHASIL (tidak diharapkan)")
except Exception as e:
  print("append kolom asing DITOLAK:", type(e).__name__)
print("versi setelah percobaan gagal:", DeltaTable(PATH_DELTA).version())

baris v0: 100000 | v1: 150000 | terbaru: 150000
selisih total_bayar (terbaru - v1): 2455.0 | jumlah baris dikoreksi: 2455
2 UPDATE
1 WRITE
0 WRITE
['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']
append kolom asing DITOLAK: SchemaMismatchError
versi setelah percobaan gagal: 2


# Small File Problem pada Tabel Delta

In [14]:
dt = DeltaTable(PATH_DELTA)
print("file data aktif sebelum:", len(dt.file_uris()), "| versi:", dt.version())

# 20 kali menambah data kecil (500 baris) - seperti data yang masuk sedikit demi sedikit
for i in range(20):
  potongan = data_delta.iloc[150_000 + i * 500: 150_000 + (i + 1) * 500]
  write_deltalake(PATH_DELTA, potongan, mode="append")
dt = DeltaTable(PATH_DELTA)
print("file data aktif setelah 20 append kecil:", len(dt.file_uris()), "| versi:", dt.version(),"| baris:", len(dt.to_pandas()))

t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sebelum = time.perf_counter()- t0

# COMPACTION: gabungkan file kecil menjadi file yang lebih besar (dicatat sebagai versi baru)
hasil = dt.optimize.compact()
print("hasil compact:", {k: hasil[k] for k in ("numFilesAdded", "numFilesRemoved")})

dt = DeltaTable(PATH_DELTA)
t0 = time.perf_counter(); _ = DeltaTable(PATH_DELTA).to_pandas(); t_sesudah = time.perf_counter()- t0
print("file data aktif setelah compact:", len(dt.file_uris()), "| versi:", dt.version(), "|baris:", len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")

# File lama TIDAK langsung terhapus (masih dibutuhkan untuk time travel). VACUUM yangmenghapusnya.
print("file parquet di disk:", len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat = dt.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True) #dry_run = hanya melaporkan
print("file yang akan dihapus VACUUM (dry run):", len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 |baris: 160000
waktu baca sebelum compact: 0.0585 s | sesudah: 0.0434 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


# Merangkum dan Memilih

In [15]:
rekap = pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv", index=False)
print(rekap.to_string(index=False))

                                            langkah  detik
                                  baca trips_bersih 0.2153
                             relasional: muat tabel 0.7460
                             key-value: bangun dict 1.7486
             cari 200 perjalanan - SQL tanpa indeks 2.6059
            cari 200 perjalanan - SQL dengan indeks 0.0023
                    cari 200 perjalanan - key-value 0.0008
                  hitung per wilayah - SQL GROUP BY 0.1239
  hitung per wilayah - key-value (buka semua nilai) 0.8537
DuckDB - hitung per wilayah (langsung dari Parquet) 0.0231
               SQLite - muat seluruh baris ke tabel 1.3238
                        SQLite - hitung per wilayah 0.3600
               DuckDB - hanya wilayah Staten Island 0.0088
                             hitung arus antar zona 0.0443
                       2 langkah - graph (NetworkX) 0.0083
                 2 langkah - SQL self-join (DuckDB) 0.0101
                      cara 1 - loop Python, 5 kotak 7.86

# Analisis Hasil

1. Pencarian satu data. Berapa kali lebih lambat tabel tanpa indeks dibanding tabel berindeks? Berapakali lebih lambat tabel berindeks dibanding key-value? Pada selisih mana perbedaan itu benar-benar berarti?

`[cari 200 perjalanan - SQL tanpa indeks] 4.4210 s`
`[cari 200 perjalanan - SQL dengan indeks] 0.0053 s`
`[cari 200 perjalanan - key-value] 0.0018 s`

Tabel tanpa indeks 834,15x lebih lambat daripada tabel berindeks karena waktu pencariannya 4.4210 s dibanding 0.053 s. Tabel berindeks 2,94x lebih lambat daripada key-value karena 0.0053 s dibanding 0.0018 s. Perbedaan paling berarti terlihat antara tabel tanpa indeks dan tabel berindeks karena selisih waktunya sangat besar.

2. Harga kesederhanaan. Berapa kali lebih lambat key-value untuk menghitung dibanding SQL GROUPBY, dan mengapa (lihat M.2)?

`[hitung per wilayah - SQL GROUP BY] 0.3811 s`
`[hitung per wilayah - key-value (buka semua nilai)] 2.3070 s`

Key-value 6,05x lebih lambat daripada SQL GROUP BY karena setiap nilai harus dibuka dengan json.loads, mengambil field yang dibutuhkan, lalu dihitung sendiri. Sementara SQL dapat melakukan agregasi langsung pada kolom angka.

3. Data fleksibel. Sebutkan satu keuntungan dan satu risiko dari kenyataan bahwa dokumen ke-20.001boleh membawa field baru.

Keuntungan: dokumen baru bisa menambahkan field tanpa harus mengubah struktur dokumen lama.
Risiko: struktur data bisa menjadi tidak konsisten karena tidak semua dokumen memiliki field yang sama.

4. Penyimpanan per kolom. Berapa kali lebih cepat DuckDB dibanding SQLite pada data Anda?Sebutkan dua penyebabnya, dan satu di antaranya yang bukan soal format kolom.

`[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.0395 s`
`[SQLite - hitung per wilayah] 0.5969 s`

DuckDB 15,11x lebih cepat daripada SQLite pada perhitungan per wilayah. Penyebabnya adalah DuckDB menggunakan penyimpanan kolumnar sehingga hanya membaca kolom yang diperlukan, serta menggunakan vectorized query execution yang yang memproses data dalam batch sehingga operasi analitik lebih efisien.

5. Graph. Bila dua langkah pada graph dan SQL sama cepatnya, apa yang membuat graph tetap layakdipilih untuk pertanyaan keterhubungan? Rujuk hasil untuk k = 1 sampai 4.

Graph tetap layak dipilih karena lebih mudah digunakan untuk mencari hubungan antar data. Pada k = 1 sampai k = 4, graph dapat mengikuti hubungan dari satu data ke data lain berdasarkan jumlah langkahnya. Walaupun pada k tertentu kecepatannya sama dengan SQL, graph lebih sederhana untuk pertanyaan yang berhubungan dengan keterhubungan data.

6. Indeks spasial. Berapa kali lebih cepat STRtree dibanding loop Python, dan dibanding NumPy?Mengapa dua angka itu berbeda jauh, dan apa artinya bagi cara Anda menulis klaim percepatan?

`[cara 1 - loop Python, 5 kotak] 13.7449 s`
`[cara 2 - NumPy, 1000 kotak] 0.5331 s`
`[cara 3 - STRtree, 1000 kotak] 0.2291 s`

STRtree sekitar 59,99x lebih cepat daripada loop Python dan 2,33x lebih cepat daripada NumPy. Angkanya berbeda jauh karena jumlah kotak yang diuji tidak sama, yaitu loop Python hanya 5 kotak, sedangkan NumPy dan STRtree 1000 kotak. Jadi, saat menulis klaim percepatan, harus disebutkan juga kondisi pengujiannya agar perbandingannya tidak menyesatkan.

7. Delta Lake. Tunjukkan dari keluaran K-7 bahwa (a) update tidak menghilangkan data lama, dan (b)penulisan yang gagal tidak meninggalkan jejak. Jelaskan juga mengapa setelah compaction file didisk masih lebih banyak daripada file aktif.

(a) Update tidak menghilangkan data lama karena jumlah baris pada v1 dan versi terbaru tetap 150.000 baris. Selain itu, versi berubah dari v1 menjadi v2, sehingga data lama tetap tersimpan sebagai versi sebelumnya.
(b) Penulisan yang gagal tidak meninggalkan jejak karena setelah percobaan append dengan kolom asing ditolak, versi Delta tetap 2. Artinya, tidak ada versi baru yang berhasil dibuat.
Setelah compaction, file di disk masih lebih banyak daripada file aktif karena file lama dari versi sebelumnya tetap disimpan untuk menjaga riwayat Delta Lake. Jadi, file aktif bisa lebih sedikit, tetapi file fisik lama masih ada di disk.

# Latihan

1. Ulangi pencarian K-2 dengan 2.000 id acak untuk tabel berindeks dan key-value saja. Hitung waktu rata-rata per pencarian dalam mikrodetik untuk masing-masing.



In [16]:
import time
import random
import json

id_uji_2000 = data["id_perjalanan"].sample(
    n=2000,
    random_state=42
).tolist()

def uji_sql_indeks():
    hasil = []
    for id_ in id_uji_2000:
        hasil.append(
            con.execute(
                "SELECT * FROM perjalanan WHERE id_perjalanan = ?",
                (id_,)
            ).fetchone()
        )
    return hasil

def uji_key_value():
    return [
        json.loads(kv[f"perjalanan:{i}"])
        for i in id_uji_2000
    ]

t0 = time.perf_counter()
uji_sql_indeks()
waktu_sql = time.perf_counter() - t0

t0 = time.perf_counter()
uji_key_value()
waktu_kv = time.perf_counter() - t0

print("SQL berindeks:")
print("total:", round(waktu_sql, 4), "s")
print("rata-rata:", round(waktu_sql / 2000 * 1_000_000, 2), "mikrodetik/pencarian")

print("\nKey-value:")
print("total:", round(waktu_kv, 4), "s")
print("rata-rata:", round(waktu_kv / 2000 * 1_000_000, 2), "mikrodetik/pencarian")

SQL berindeks:
total: 0.0231 s
rata-rata: 11.57 mikrodetik/pencarian

Key-value:
total: 0.0078 s
rata-rata: 3.91 mikrodetik/pencarian


2. Pada K-3, hitung dokumen dengan zona tujuan tertentu dan metode_bayar "Tunai". Hitung juga angka yang sama lewat SQL pada tabel di K-2. Apakah sama? Bila tidak, apa sebabnya?

In [17]:
zona_uji = int(sample["zona_tujuan"].iloc[0])

jumlah_tinydb = len(
    tabel.search(
        (Q.tujuan.zona == zona_uji) &
        (Q.biaya.metode_bayar == "Tunai")
    )
)

print("Zona tujuan yang diuji:", zona_uji)
print("Jumlah TinyDB:", jumlah_tinydb)

jumlah_sql = con.execute("""
    SELECT COUNT(*)
    FROM perjalanan
    WHERE zona_tujuan = ?
      AND metode_bayar = 'Tunai'
""", (zona_uji,)).fetchone()[0]

print("Jumlah SQL:", jumlah_sql)

print("Hasil sama?", jumlah_tinydb == jumlah_sql)

Zona tujuan yang diuji: 231
Jumlah TinyDB: 90
Jumlah SQL: 1361
Hasil sama? False


Hasilnya tidak sama yaitu TinyDB = 90 dokumen dan SQL = 1361 baris. Perbedaan tersebut terjadi karena jumlah data pada kedua penyimpanan berbeda. K-3 hanya menggunakan 20.000 dokumen dari sample_head(20_000) kemudian ditambah satu dokumen baru sehingga menjadi 20.001 dokumen, sedangkan tabel SQL pada K-2 berisi sekitar 300.000 baris. Karena jumlah data yang diperiksa berbeda, hasil perhitungannya juga berbeda.

3. Pada K-4, jalankan SUM atas satu kolom angka, lalu SUM atas beberapa kolom angka sekaligus. Bandingkan waktunya dan jelaskan dengan column pruning (Praktikum 3/K-4).

In [18]:
waktu_mulai = time.perf_counter()

hasil_1 = con_d.sql("""
    SELECT SUM(total_bayar)
    FROM perjalanan
""").fetchone()[0]

waktu_1 = time.perf_counter() - waktu_mulai

waktu_mulai = time.perf_counter()

hasil_3 = con_d.sql("""
    SELECT
        SUM(total_bayar),
        SUM(jarak_mil),
        SUM(zona_jemput)
    FROM perjalanan
""").fetchone()

waktu_3 = time.perf_counter() - waktu_mulai


print("SUM 1 kolom:")
print("total_bayar =", hasil_1)
print("waktu =", round(waktu_1, 4), "s")

print("\nSUM 3 kolom:")
print("total_bayar =", hasil_3[0])
print("jarak_mil =", hasil_3[1])
print("zona_jemput =", hasil_3[2])
print("waktu =", round(waktu_3, 4), "s")

print("\nPerbandingan:",
      round(waktu_3 / waktu_1, 2), "x")

SUM 1 kolom:
total_bayar = 18267397.280000076
waktu = 0.0259 s

SUM 3 kolom:
total_bayar = 18267397.280000046
jarak_mil = 3203329.819999964
zona_jemput = 146371621
waktu = 0.2701 s

Perbandingan: 10.42 x


SUM pada satu kolom membutuhkan waktu 0.0259 detik, sedangkan SUM pada tiga kolom membutuhkan waktu 0.2701 detik. Jadi, perhitungan pada tiga kolom sekitar 10.42x lebih lama. Hal ini berkaitan dengan column pruning, yaitu DuckDB hanya membaca kolom yang diperlukan oleh query. Pada SUM satu kolom, hanya satu kolom yang perlu dibaca, sedangkan pada SUM beberapa kolom, lebih banyak kolom harus dibaca sehingga waktu pemrosesan menjadi lebih lama.

4. Pada K-5, cari lima zona dengan jumlah zona tujuan berbeda terbanyak menggunakan graph (out-degree), lalu silangkan dengan SQL memakai COUNT(DISTINCT tujuan).

In [19]:
hasil_graph = []

for zona in G.nodes:
    jumlah_tujuan = G.out_degree(zona)
    hasil_graph.append((zona, jumlah_tujuan))

hasil_graph = sorted(
    hasil_graph,
    key=lambda x: (-x[1], x[0])
)[:5]

print("5 zona dengan tujuan berbeda terbanyak (Graph):")
for zona, jumlah in hasil_graph:
    print(zona, jumlah)

hasil_sql = con_d.sql("""
    SELECT
        zona_jemput AS zona,
        COUNT(DISTINCT zona_tujuan) AS jumlah_tujuan
    FROM perjalanan
    GROUP BY zona_jemput
    ORDER BY jumlah_tujuan DESC, zona
    LIMIT 5
""").fetchall()

print("\n5 zona dengan tujuan berbeda terbanyak (SQL):")
for zona, jumlah in hasil_sql:
    print(zona, jumlah)

print("\nHasil sama?", hasil_graph == hasil_sql)

5 zona dengan tujuan berbeda terbanyak (Graph):
30 263
46 263
52 263
55 263
74 263

5 zona dengan tujuan berbeda terbanyak (SQL):
30 263
46 263
52 263
55 263
74 263

Hasil sama? True


Lima zona dengan jumlah tujuan berbeda terbanyak berdasarkan graph adalah zona 30, 46, 52, 55, dan 74 masing-masing memiliki 263 tujuan berbeda. Hasil tersebut sama dengan perhitungan SQL menggunakan `COUNT(DISTINCT zona_tujuan)`, sehingga dapat disimpulkan bahwa hasil graph dan SQL konsisten.

5. Pada K-6, ulangi pengukuran dengan sisi kotak 0,02 derajat dan 0,001 derajat. Bagaimana perbandingan STRtree terhadap NumPy berubah? Mengapa kotak yang lebih kecil lebih menguntungkan indeks?

In [20]:
for sisi in [0.02, 0.001]:

    kotak_uji = [
        box(x - sisi/2, y - sisi/2, x + sisi/2, y + sisi/2)
        for x, y in zip(cx, cy)
    ]

    waktu_mulai = time.perf_counter()

    hasil_np = []
    for x, y in zip(cx, cy):
        m = (
            (bujur >= x - sisi/2) &
            (bujur <= x + sisi/2) &
            (lintang >= y - sisi/2) &
            (lintang <= y + sisi/2)
        )
        hasil_np.append(int(m.sum()))

    waktu_np = time.perf_counter() - waktu_mulai

    waktu_mulai = time.perf_counter()

    hasil_tree = [
        len(pohon.query(k, predicate="contains"))
        for k in kotak_uji
    ]

    waktu_tree = time.perf_counter() - waktu_mulai

    print(f"\nSisi kotak: {sisi} derajat")
    print(f"NumPy   : {waktu_np:.4f} s")
    print(f"STRtree : {waktu_tree:.4f} s")
    print(f"STRtree vs NumPy: {waktu_np / waktu_tree:.2f}x lebih cepat")
    print("Hasil sama?", hasil_np == hasil_tree)


Sisi kotak: 0.02 derajat
NumPy   : 0.2385 s
STRtree : 0.6221 s
STRtree vs NumPy: 0.38x lebih cepat
Hasil sama? True

Sisi kotak: 0.001 derajat
NumPy   : 0.2101 s
STRtree : 0.0093 s
STRtree vs NumPy: 22.71x lebih cepat
Hasil sama? True


Pada kotak berukuran 0.02°, STRtree membutuhkan 0.6221 detik, sedangkan NumPy 0.2385 detik sehingga STRtree lebih lambat. Namun pada kotak 0.001°, STRtree hanya membutuhkan 0.0093 detik dan menjadi 22.71x lebih cepat daripada NumPy. Hasil kedua metode tetap sama (True). Kotak yang lebih kecil lebih menguntungkan STRtree karena indeks spasial dapat menyaring lebih banyak titik yang berada di luar area pencarian sehingga kandidat yang perlu diperiksa menjadi jauh lebih sedikit.

6. Jalankan sel di bawah setelah K-8. Jelaskan perbedaan time travel (membaca versi lama) dengan restore (mengembalikan tabel ke versi lama), dan mengapa restore dicatat sebagai versi baru, bukan menghapus riwayat.

In [21]:
dt = DeltaTable(PATH_DELTA)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))
dt.restore(1)                   # kembalikan tabel ke keadaan versi 1 (dicatat sebagai versi baru)
dt = DeltaTable(PATH_DELTA)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


Time travel digunakan untuk membaca atau mengakses data pada versi lama tanpa mengubah versi terbaru tabel. Sedangkan restore digunakan untuk mengembalikan isi tabel ke kondisi versi tertentu. Pada percobaan, sebelum restore tabel berada di versi 23 dengan 160.000 baris, kemudian dilakukan restore(1) sehingga menjadi versi 24 dengan 150.000 baris. Restore dicaatat sebagai versi baru agar seluruh riwayat perubahan tetap tersimpan dan versi sebelumnya masih dapat dilacak atau diakses kembali. Jadi, restore tidak menghapus riwayat, melainkan menambahkan perubahan baru yang mengembalikan keadaan tabel ke versi lama.

# Tugas Praktikum

1. Rekap benchmark. Kumpulkan benchmark_nosql.csv dan tabel rekap K-9 yang sudah terisi, dengansatu kalimat interpretasi untuk tiap baris.



```
                                            langkah  detik
                                  baca trips_bersih 0.2153
                             relasional: muat tabel 0.7460
                             key-value: bangun dict 1.7486
             cari 200 perjalanan - SQL tanpa indeks 2.6059
            cari 200 perjalanan - SQL dengan indeks 0.0023
                    cari 200 perjalanan - key-value 0.0008
                  hitung per wilayah - SQL GROUP BY 0.1239
  hitung per wilayah - key-value (buka semua nilai) 0.8537
DuckDB - hitung per wilayah (langsung dari Parquet) 0.0231
               SQLite - muat seluruh baris ke tabel 1.3238
                        SQLite - hitung per wilayah 0.3600
               DuckDB - hanya wilayah Staten Island 0.0088
                             hitung arus antar zona 0.0443
                       2 langkah - graph (NetworkX) 0.0083
                 2 langkah - SQL self-join (DuckDB) 0.0101
                      cara 1 - loop Python, 5 kotak 7.8672
                         cara 2 - NumPy, 1000 kotak 0.2388
                              bangun indeks STRtree 0.0466
                       cara 3 - STRtree, 1000 kotak 0.1596
```




Interpretasi tiap baris


1.   baca trips_bersih: pembacaan dataset Parquet membutuhkan 0.2153 detik dan menjadi waktu dasar untuk memuat data.
2.   relasional: muat tabel: memuat data ke tabel relasional membutuhkan 0.7460 detik karena seluruh data perlu ditulis ke SQLite.
3. key-value: bangun dict: membangun struktur key-value membutuhkan 1.7486 detik karena setiap baris diubah menjadi pasangan key-value.
4. cari 200 perjalanan - SQL tanpa indeks: pencarian tanpa indeks membutuhkan 2.6059 detik karena database harus mencari tanpa bantuan indeks.
5. cari 200 perjalanan - SQL dengan indeks: penggunaan indeks membuat pencarian 200 perjalanan menjadi jauh lebih cepat.
6. cari 200 perjalanan - key-value: key-value menjadi yang tercepat untuk pencarian langsung berdasarkan ID pada pengujian ini.
7. hitung per wilayah - SQL GROUP BY: SQL GROUP BY dapat melakukan agregasi per wilayah secara relatif efisien.
8. hitung per wilayah - key-value (buka semua nilai): key-value lebih lambat untuk agregasi karena seluruh nilai harus dibuka dan diproses kembali.
9. DuckDB - hitung per wilayah (langsung dari Parquet): DuckDB sangat cepat melakukan agregasi langsung dari Parquet tanpa perlu memuat seluruh data ke tabel relasional.
10. SQLite - muat seluruh baris ke tabel: memuat seluruh baris ke SQLite membutuhkan waktu 1.3238 detik karena data harus ditulis ke tabel.
11. SQLite - hitung per wilayah: agregasi per wilayah menggunakan SQLite membutuhkan waktu lebih lama daripada DuckDB pada pengujian ini.
12. DuckDB - hanya wilayah Staten Island: filter untuk satu wilayah dapat dilakukan sangat cepat oleh DuckDB karena hanya data yang diperlukan yang diproses.
13. hitung arus antar zona: perhitungan jumlah perjalanan antar pasangan zona membutuhkan 0.0443 detik untuk membentuk data arus graph.
14. 2 langkah - graph (NetworkX): graph dapat menemukan zona yang terjangkau dalam dua langkah deengan waktu 0.0083 detik.
15. 2 langkah - SQL self-join (DuckDB): SQL self-join menghasilkan pencarian dua langkah yang sedikit lebih lambat daripada graph pada pengujian ini.
16. cara 1 - loop Python, 5 kotak: loop Python membutuhkan waktu paling lama karena memeriksa titik satu per satu.
17. cara 2 - NumPy, 1000 kotak: NumPy jauh lebih cepat daripada loop karena perbandingan titik dilakukan secara vektorisasi.
18. bangun indeks STRtree: pembangunan indeks STRtree membutuhkan 0.0466 detik sebelum indeks dapat digunakan untuk pencarian spasial.
19. cara 3 - STRtree, 1000 kotak: STRtree lebih cepat daripada NumPy pada pengukuran awal karena indeks spasial mengurangi jumlah kandidat yang perlu diperiksa.



Delta Lake dua hari. Simulasikan dua hari kerja: hari pertama menulis sebagian data, hari keduamenambah data dan mengoreksi beberapa baris. Tunjukkan bahwa Anda bisa membaca tabel sepertikeadaannya di akhir hari pertama, dan lampirkan isi _delta_log/.

In [22]:
from deltalake import write_deltalake, DeltaTable
import os
import shutil
import pandas as pd

PATH_DELTA_2HARI = "/content/trips_delta_2hari"

if os.path.exists(PATH_DELTA_2HARI):
    shutil.rmtree(PATH_DELTA_2HARI)

data_delta = sample.copy()

data_delta["id_perjalanan"] = data_delta["id_perjalanan"].astype("int64")
data_delta["total_bayar"] = data_delta["total_bayar"].astype("float64")

hari1 = data_delta.iloc[:100_000].copy()

write_deltalake(
    PATH_DELTA_2HARI,
    hari1,
    mode="overwrite"
)

dt = DeltaTable(PATH_DELTA_2HARI)

print("=== AKHIR HARI 1 ===")
print("Versi:", dt.version())
print("Jumlah baris:", len(dt.to_pandas()))

hari2 = data_delta.iloc[100_000:150_000].copy()

write_deltalake(
    PATH_DELTA_2HARI,
    hari2,
    mode="append"
)

dt = DeltaTable(PATH_DELTA_2HARI)

print("\n=== SETELAH DATA HARI 2 DITAMBAHKAN ===")
print("Versi:", dt.version())
print("Jumlah baris:", len(dt.to_pandas()))

dt.update(
    predicate="id_perjalanan >= 1 AND id_perjalanan <= 2455",
    updates={
        "total_bayar": "total_bayar + 1.0"
    }
)

dt = DeltaTable(PATH_DELTA_2HARI)

print("\n=== SETELAH KOREKSI HARI 2 ===")
print("Versi:", dt.version())
print("Jumlah baris:", len(dt.to_pandas()))

=== AKHIR HARI 1 ===
Versi: 0
Jumlah baris: 100000

=== SETELAH DATA HARI 2 DITAMBAHKAN ===
Versi: 1
Jumlah baris: 150000

=== SETELAH KOREKSI HARI 2 ===
Versi: 2
Jumlah baris: 150000


In [23]:
dt_hari1 = DeltaTable(
    PATH_DELTA_2HARI,
    version=0
)

data_hari1 = dt_hari1.to_pandas()

print("=== TIME TRAVEL: AKHIR HARI 1 ===")
print("Versi:", dt_hari1.version())
print("Jumlah baris:", len(data_hari1))

display(data_hari1.head())

=== TIME TRAVEL: AKHIR HARI 1 ===
Versi: 0
Jumlah baris: 100000


,waktu_jemput,tpep_dropoff_datetime,passenger_count,jarak_mil,zona_jemput,zona_tujuan,payment_type,fare_amount,tip,total_bayar,durasi_menit,metode_bayar,suhu_c,hujan_mm,wilayah_jemput,id_perjalanan
0,2023-01-21 20:45:00,2023-01-21 21:20:00,4.0,5.17,231,231,1,18.20,0.73,21.93,35.0,Kartu kredit,9.1,0.19,Manhattan,1
1,2023-01-02 16:19:00,2023-01-02 17:06:00,2.0,2.41,160,177,2,14.16,0.24,17.40,47.0,Tunai,5.3,0.00,Queens,2
2,2023-01-29 13:19:00,2023-01-29 14:24:00,4.0,1.14,144,252,1,7.00,2.95,12.95,65.0,Kartu kredit,1.4,0.17,Brooklyn,3
3,2023-01-19 00:39:00,2023-01-19 01:05:00,3.0,2.23,138,183,1,9.20,1.11,13.31,26.0,Kartu kredit,9.3,0.08,Brooklyn,4
4,2023-01-28 10:40:00,2023-01-28 10:47:00,1.0,1.36,143,166,2,8.59,0.66,12.25,7.0,Tunai,5.9,0.17,Staten%20Island,5


In [24]:
import glob
import os

LOG_DIR = os.path.join(PATH_DELTA_2HARI, "_delta_log")

print("=== ISI _delta_log/ ===")

for f in sorted(glob.glob(os.path.join(LOG_DIR, "*"))):
    print(os.path.basename(f))

=== ISI _delta_log/ ===
00000000000000000000.json
00000000000000000001.json
00000000000000000002.json
